# Fine-tuning transformer untuk klasifikasi tema hadis (JADS #1622, revisi tahap 2)

Menjawab komentar R1-6, R2-5, R3-5: pembanding transformer **yang di-fine-tune** pada **lipatan yang sama** dengan nested CV TF-IDF (1.323 hadis).

**Sebelum mulai**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Siapkan berkas `hasil_langkah5_1323_v2.zip` (dari chat) untuk diunggah di sel 3.
3. Jalankan sel dari atas ke bawah. Perkiraan waktu di T4: sekitar 5 menit per model per ulangan. Dengan 3 model dan 3 ulangan sekitar 45 menit.

**Catatan desain**: hiperparameter dibuat tetap (tanpa penyetelan memakai data uji), jumlah epoch tetap, dan prediksi disimpan dalam format yang sama dengan `oof_all.csv` agar uji berpasangan bisa dihitung ulang di sisi analisis.

## 1. Pemasangan dan pemeriksaan GPU

In [ ]:
!pip -q install transformers accelerate
import torch, platform
print("torch", torch.__version__, "| GPU tersedia:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print("PERINGATAN: tanpa GPU proses akan sangat lambat. Aktifkan T4 GPU lalu mulai ulang.")

## 2. Data: ambil dari GitHub

In [ ]:
!git clone -q --depth 1 https://github.com/Sintia-Afriyani/hadith-theme-classification-UAD /content/repo
import pandas as pd
d = pd.read_csv("/content/repo/dataset_rilis_tanpa_enam.csv").sort_values("url", kind="stable").reset_index(drop=True)
d.insert(0, "rec_id", [f"H{i+1:04d}" for i in range(len(d))])
print(len(d), d.label.value_counts().to_dict())

## 3. Unggah zip hasil (berisi `fold_assignments.csv` dan `dataset_ids_v1.1.csv`)

In [ ]:
from google.colab import files
import zipfile, io, os
up = files.upload()                       # pilih hasil_langkah5_1323_v2.zip
name = next(iter(up))
os.makedirs("/content/hasil", exist_ok=True)
zipfile.ZipFile(io.BytesIO(up[name])).extractall("/content/hasil")
base = "/content/hasil/out"
folds = pd.read_csv(f"{base}/fold_assignments.csv")
ids = pd.read_csv(f"{base}/dataset_ids_v1.1.csv")
assert len(folds) == len(d) == 1323
assert (folds.rec_id.values == d.rec_id.values).all() and (folds.url.values == d.url.values).all(), "urutan/ID tidak cocok"
assert (ids.url.values == d.url.values).all()
print("ID dan lipatan cocok. Kolom ulangan:", [c for c in folds.columns if c.startswith("rep")])

## 4. Pengaturan

In [ ]:
REPS   = [0, 1, 2]                      # ulangan (dari 10 ulangan nested CV) yang dijalankan
MODELS = {                                # nama pendek -> ID Hugging Face
    "BERT-base (fine-tuned)":        "bert-base-uncased",
    "RoBERTa-base (fine-tuned)":     "roberta-base",
    "mBERT (fine-tuned)":            "bert-base-multilingual-cased",
    "XLM-RoBERTa-base (fine-tuned)": "xlm-roberta-base",
}
HP = dict(lr=3e-5, epochs=4, batch=16, maxlen=256, warmup=0.1, wd=0.01, seed=42)   # sama untuk semua model
QUICK = False                             # True = uji cepat 1 lipatan, 1 epoch (HASIL TIDAK UNTUK DILAPORKAN)
print(HP)

## 5. Fungsi pelatihan

In [ ]:
import numpy as np, torch, time, random
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

classes = sorted(d.label.unique()); c2i = {c: i for i, c in enumerate(classes)}
y_all = np.array([c2i[c] for c in d.label]); X_all = d.text.tolist()
dev = "cuda" if torch.cuda.is_available() else "cpu"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def encode(tok, texts):
    return tok(texts, truncation=True, max_length=HP["maxlen"])

def batches(enc, y, idx, bs, shuffle, tok):
    order = np.random.permutation(idx) if shuffle else idx
    for i in range(0, len(order), bs):
        b = order[i:i+bs]
        feats = [{k: enc[k][j] for k in enc.keys()} for j in b]
        out = tok.pad(feats, return_tensors="pt")
        out["labels"] = torch.tensor(y[b])
        yield {k: v.to(dev) for k, v in out.items()}

def run_fold(model_id, tr, te, tok_cache={}):
    seed_all(HP["seed"])
    tok = tok_cache.get(model_id) or tok_cache.setdefault(model_id, AutoTokenizer.from_pretrained(model_id))
    enc = encode(tok, X_all)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=len(classes)).to(dev)
    epochs = 1 if QUICK else HP["epochs"]
    steps = epochs * int(np.ceil(len(tr) / HP["batch"]))
    nd = ["bias", "LayerNorm.weight"]
    params = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nd)], "weight_decay": HP["wd"]},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in nd)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=HP["lr"])
    sch = get_linear_schedule_with_warmup(opt, int(HP["warmup"] * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))
    model.train()
    for ep in range(epochs):
        tot = 0.0
        for b in batches(enc, y_all, tr, HP["batch"], True, tok):
            with torch.autocast(device_type=dev, dtype=torch.float16, enabled=(dev == "cuda")):
                loss = model(**b).loss
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
    model.eval(); preds = []
    with torch.no_grad():
        for b in batches(enc, y_all, te, 64, False, tok):
            b.pop("labels"); preds.append(model(**b).logits.argmax(-1).cpu().numpy())
    del model, opt; torch.cuda.empty_cache()
    return np.concatenate(preds), tot

## 6. Jalankan (hasil disimpan setiap selesai satu model per ulangan)

In [ ]:
import os
rows, oof = [], []
os.makedirs("/content/out_bert", exist_ok=True)
t0 = time.time()
for rep in REPS:
    fa = folds[f"rep{rep}"].values
    for name, mid in MODELS.items():
        for fold in range(5):
            if QUICK and fold > 0: break
            tr, te = np.where(fa != fold)[0], np.where(fa == fold)[0]
            t1 = time.time(); p, last_loss = run_fold(mid, tr, te); sec = time.time() - t1
            yt = y_all[te]
            rows.append(dict(model=name, hf_id=mid, repeat=rep, fold=fold, accuracy=accuracy_score(yt, p),
                             macro_f1=f1_score(yt, p, average="macro"), balanced_acc=balanced_accuracy_score(yt, p),
                             fit_s=sec, n_test=len(te), **{k: v for k, v in HP.items()}))
            oof.append(pd.DataFrame({"rec_id": d.rec_id.values[te], "repeat": rep, "fold": fold, "model": name,
                                     "true": [classes[i] for i in yt], "pred": [classes[i] for i in p]}))
            print(f"ulangan {rep} | {name:30s} | lipatan {fold} | acc {rows[-1]['accuracy']:.3f} | F1 {rows[-1]['macro_f1']:.3f} | {sec:.0f} dtk | total {(time.time()-t0)/60:.1f} mnt", flush=True)
        pd.DataFrame(rows).to_csv("/content/out_bert/bert_cv_per_fold.csv", index=False)
        pd.concat(oof).to_csv("/content/out_bert/bert_oof.csv", index=False)
print("SELESAI")

## 7. Ringkasan dan perbandingan dengan ensemble TF-IDF pada lipatan yang sama

In [ ]:
from scipy import stats
res = pd.DataFrame(rows)
tf = pd.read_csv(f"{base}/cv_per_fold.csv")
tf = tf[tf["repeat"].isin(REPS)]
out = []
ref = tf[tf.model == "Ensemble (tuned, nested CV)"].sort_values(["repeat", "fold"])
for name in list(MODELS) + ["Ensemble (tuned, nested CV)", "SVM (default)"]:
    r = res[res.model == name] if name in MODELS else tf[tf.model == name]
    r = r.sort_values(["repeat", "fold"])
    row = dict(model=name, n_fold=len(r), akurasi=r.accuracy.mean(), akurasi_sd=r.accuracy.std(ddof=1),
               macro_f1=r.macro_f1.mean(), macro_f1_sd=r.macro_f1.std(ddof=1), detik_per_lipatan=r.fit_s.mean())
    if name in MODELS and len(r) == len(ref):
        dif = r.macro_f1.values - ref.macro_f1.values; n = len(dif)
        row["selisih_vs_ensemble"] = dif.mean()
        row["p_Nadeau_Bengio"] = 2 * (1 - stats.t.cdf(abs(dif.mean() / np.sqrt((1/n + 0.25) * dif.var(ddof=1))), n - 1))
    out.append(row)
summary = pd.DataFrame(out); summary.to_csv("/content/out_bert/bert_ringkasan.csv", index=False)
display(summary.round(4))

## 8. Unduh hasil (kirim zip ini ke chat)

In [ ]:
import shutil
shutil.make_archive("/content/hasil_bert", "zip", "/content/out_bert")
files.download("/content/hasil_bert.zip")